# Influence maximization with GIP and NaDS

This notebook selects a fixed number of municipalities to maximize population-weighted generalized influence propagation. Hyperedges use real company-size parameters and nodes use annual population weights mapped through the BDAP municipality registry. The NaDS search has no MG phase and imposes no connectivity requirement on the seed set.

In [17]:
from pathlib import Path
import sys

import hypernetx as hnx
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from the project or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

from src.gip_model import (
    GIPParameters,
    aligned_edge_parameters,
    aligned_node_weights,
    gip,
)
from src.nads import nads

In [18]:
# Data and graph configuration
YEAR = 2018
SEED_BUDGET = 10
MIN_EDGE_SIZE = 2
DATA_DIR = PROJECT_ROOT / "data" / "processed"
PARAMETER_FILE = DATA_DIR / "hyperedge_parameters.csv"
NODE_PARAMETER_FILE = DATA_DIR / "node_parameters.csv"

# Edge-parameter configuration
EDGE_WEIGHT_MEAN = 1.0
EDGE_WEIGHT_BETA = 1.0  # 0 = all equal, 0.5 = softer, 1 = full heterogeneity

# Population-derived node-weight configuration
NODE_WEIGHT_MEAN = 1.0
NODE_WEIGHT_BETA = 1.0  # 0 = all equal; 1 = full relative population differences

# GIP configuration
GIP_PARAMS = GIPParameters(
    h0=1.0,
    l0=1.0,
    theta_l=2.0,
    theta_h=50.0,
    gamma=0.1,
    eps=0.01,
)
ALPHA = 0.1
STRESS_LEVEL = 5.0
SEED_INTENSITY = 1.0

# NaDS configuration
DELTA = 0.5
XI = 0.01
D = 2  # unrestricted one-for-one exchanges
SEARCH_SECONDS = 600
MAX_NEIGHBORS_PER_PHASE = None
RANDOM_SEED = 42

In [19]:
incidences = pd.read_csv(DATA_DIR / f"rete_{YEAR}.csv")
edge_sizes = incidences.groupby("CF Partecipata")["CF Comune"].nunique()
valid_edges = edge_sizes.loc[edge_sizes.ge(MIN_EDGE_SIZE)].index
filtered = incidences.loc[incidences["CF Partecipata"].isin(valid_edges)].copy()

H = hnx.Hypergraph(
    filtered,
    edge_col="CF Partecipata",
    node_col="CF Comune",
    cell_weight_col="Quota",
)

# Keep the incidence matrix sparse: NaDS evaluates the GIP objective many times.
incidence_matrix = H.incidence_matrix(weights="weight")
edge_weights, edge_parameter_audit = aligned_edge_parameters(
    H.edges,
    YEAR,
    PARAMETER_FILE,
    target_mean=EDGE_WEIGHT_MEAN,
    beta=EDGE_WEIGHT_BETA,
)
node_weights, node_parameter_audit = aligned_node_weights(
    H.nodes,
    YEAR,
    NODE_PARAMETER_FILE,
    target_mean=NODE_WEIGHT_MEAN,
    beta=NODE_WEIGHT_BETA,
)

print(f"Nodes: {len(H.nodes):,}")
print(f"Hyperedges: {len(H.edges):,}")
print(f"Incidences: {len(filtered):,}")
display(pd.Series(edge_weights, name="edge_parameter").describe())
display(edge_parameter_audit.head())
display(pd.Series(node_weights, name="population_node_weight").describe())
display(node_parameter_audit["node_weight_basis"].value_counts())
display(node_parameter_audit.head())

Nodes: 7,781
Hyperedges: 4,501
Incidences: 134,504


count    4501.000000
mean        1.000000
std         0.487738
min         0.129295
25%         0.594167
50%         0.924909
75%         1.383380
max         1.997921
Name: edge_parameter, dtype: float64

,year,company_name,edge_parameter_base_0_1,edge_parameter_basis,company_size_score_0_100,company_size_data_confidence_0_1,company_size_data_quality,edge_parameter
company_id,,,,,,,,
2171040807,2018,"PROMEDIA, SOC. CONSORTILE R.L.",0.173775,company_size_direct,17.377483,1.0000,low,0.347414
2389030798,2018,PIANA AMBIENTE S.P.A. IN LIQUIDAZIONE,0.503983,company_size_confidence_shrunk,54.331429,0.5125,low,1.007573
857000947,2018,"S.F.I.D.E. - SVILUPPO, FORMAZIONE, IDEAZIONE D...",0.199325,company_size_direct,19.932523,1.0000,high,0.398495
947590949,2018,G.A.L. MOLISE RURALE - SOCIETA' CONSORTILE A R...,0.287913,company_size_confidence_shrunk,19.383242,0.6500,medium,0.575602
1622170700,2018,AGENZIA DI SVILUPPO RURALE MOLISE GRUPPO DI AZ...,0.211931,company_size_confidence_shrunk,19.160335,0.9250,high,0.423696


count    7781.000000
mean        1.000000
std         5.676238
min         0.004243
25%         0.139220
50%         0.327763
75%         0.820069
max       373.932947
Name: population_node_weight, dtype: float64

node_weight_basis
population_direct            7594
year_median_no_bdap_match     187
Name: count, dtype: int64

,year,municipality_tax_code,municipality_bdap_id,municipality_name,population,population_data_available,node_weight_population,node_weight_basis,node_weight
municipality_id,,,,,,,,,
8010803,2018,00008010803,368542930529988102,COMUNE DI CINQUEFRONDI,6534.0,True,6534.0,population_direct,0.866343
31500945,2018,00031500945,329542930459791702,COMUNE DI COLLI A VOLTURNO,1340.0,True,1340.0,population_direct,0.177671
31730948,2018,00031730948,183742930525710802,COMUNE DI SANTA MARIA DEL MOLISE,678.0,True,678.0,population_direct,0.089896
33120437,2018,00033120437,778542930519394202,COMUNE DI MATELICA,9617.0,True,9617.0,population_direct,1.275118
34670943,2018,00034670943,387242928779095702,COMUNE DI ISERNIA,21400.0,True,21400.0,population_direct,2.837427


In [20]:
def influence_objective(seed_indicator: np.ndarray) -> float:
    result = gip(
        incidence_matrix,
        edge_weights,
        seed_indicator * SEED_INTENSITY,
        GIP_PARAMS,
        node_weights=node_weights,
        stress_level=STRESS_LEVEL,
        alpha=ALPHA,
    )
    return result.total_spread

In [21]:
rng = np.random.default_rng(RANDOM_SEED)
initial_seed_indicator = np.zeros(len(H.nodes), dtype=float)
initial_seed_indices = rng.choice(
    len(H.nodes), size=SEED_BUDGET, replace=False
)
initial_seed_indicator[initial_seed_indices] = 1.0

print(f"Initial objective: {influence_objective(initial_seed_indicator):.6g}")

Initial objective: 3797.63


In [22]:
spread_history, seed_history, evaluation_history = nads(
    objective=influence_objective,
    x0=initial_seed_indicator,
    delta=DELTA,
    xi=XI,
    d=D,
    max_time=SEARCH_SECONDS,
    buffer_dim=10_000,
    max_neighbors_per_phase=MAX_NEIGHBORS_PER_PHASE,
    random_seed=RANDOM_SEED,
    verbose=1,
)

NaDS done: calls=474459, time=600.0/600.0s, spread=25755.2            


In [23]:
best_seed_indicator = seed_history[-1]
node_order = np.asarray(list(H.nodes))
selected_ids = node_order[np.flatnonzero(best_seed_indicator)]

municipalities = (
    node_parameter_audit
    .reindex(selected_ids)[
        ["municipality_name", "population", "node_weight", "node_weight_basis"]
    ]
    .reset_index()
)

print(f"Initial spread: {spread_history[0]:.6g}")
print(f"Best spread: {spread_history[-1]:.6g}")
print(f"Accepted improvements: {len(spread_history) - 1}")
print(f"Selected seeds: {np.count_nonzero(best_seed_indicator)}")
display(municipalities)

evaluation_table = pd.DataFrame(
    evaluation_history,
    columns=["spread", "elapsed_seconds", "objective_calls"],
)
display(evaluation_table.tail(20))

Initial spread: 3797.63
Best spread: 25755.2
Accepted improvements: 40
Selected seeds: 10


,municipality_id,municipality_name,population,node_weight,node_weight_basis
0,137020871,COMUNE DI CATANIA,297752.0,39.478949,population_direct
1,215150236,COMUNE DI VERONA,258584.0,34.285663,population_direct
2,514490010,COMUNE DI TORINO,860793.0,114.132578,population_direct
3,1199250158,COMUNE DI MILANO,1395980.0,185.093043,population_direct
4,1232710374,COMUNE DI BOLOGNA,393248.0,52.140768,population_direct
5,1307110484,COMUNE DI FIRENZE,369885.0,49.043067,population_direct
6,2438750586,ROMA CAPITALE,2820219.0,373.932947,population_direct
7,80000350654,COMUNE DI CAVA DE' TIRRENI,51494.0,6.827591,population_direct
8,80015010723,COMUNE DI BARI,316491.0,41.963554,population_direct
9,80016350821,COMUNE DI PALERMO,652720.0,86.544170,population_direct


,spread,elapsed_seconds,objective_calls
58,24066.939350,77.610065,75103
59,24145.030329,81.252208,78727
60,24356.416064,83.869505,81354
61,24368.403781,83.892947,81377
62,24566.048264,91.542266,88683
63,24576.031629,104.234704,100400
64,24631.266567,165.192062,150851
65,24850.882991,165.701726,151283
66,24889.473591,206.368602,182266
67,24891.958209,226.148421,197881


In [24]:
best_diffusion = gip(
    incidence_matrix,
    edge_weights,
    best_seed_indicator * SEED_INTENSITY,
    GIP_PARAMS,
    node_weights=node_weights,
    stress_level=STRESS_LEVEL,
    alpha=ALPHA,
)

diffusion_table = pd.DataFrame({
    "step": range(len(best_diffusion.states)),
    "active_municipalities": [
        np.count_nonzero(state) for state in best_diffusion.states
    ],
    "cumulative_spread": best_diffusion.spread_history,
})
display(diffusion_table)

,step,active_municipalities,cumulative_spread
0,0,10,983.442330
1,1,1968,17719.695680
2,2,5966,24017.307540
3,3,6967,25466.178770
4,4,7017,25755.228537
5,5,0,25755.228537
